# Classification with MLP Neural Network (Multilayer Perceptron)

## What is a Neural Network

An **artificial neural network** is a computational model inspired by the functioning of the human brain. It is composed of basic units called **artificial neurons** (or nodes), organized in **layers** that process information sequentially.

An MLP (Multilayer Perceptron) is the most fundamental neural network architecture. It consists of:

- An **input layer**: receives the raw data (the features)
- One or more **hidden layers**: learn intermediate representations of the data
- An **output layer**: produces the final prediction

<mark>The MLP is the ideal starting point for understanding neural networks, as all essential concepts — neuron, weight, bias, activation, backpropagation — are present in a direct and transparent way.</mark>

### The artificial neuron

Each neuron performs a simple two-step operation:

**1. Linear combination** — multiplies each input by its weight and adds a bias:

$$z = w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b = \mathbf{w} \cdot \mathbf{x} + b$$

**2. Activation function** — applies a non-linear transformation to the result:

$$\hat{y} = f(z)$$

The **weights** and **bias** are the parameters the network learns during training. The **activation function** is what allows the network to learn non-linear relationships in the data — without it, stacking layers would be pointless (the result would always be a linear function).

### Activation functions

The most common activation functions in modern neural networks are:

| Function | Formula | Typical use |
|----------|---------|-------------|
| **ReLU** | $f(z) = \max(0, z)$ | Hidden layers (current standard) |
| **Sigmoid** | $f(z) = \frac{1}{1 + e^{-z}}$ | Binary output (probability) |
| **Softmax** | $f(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$ | Multi-class output |
| **Tanh** | $f(z) = \tanh(z)$ | Recurrent layers |

In this notebook we will use **ReLU** in the hidden layers — it is simple, efficient, and solves the *vanishing gradient* problem that affects Sigmoid in deep networks.

### How the network learns — Backpropagation

Neural network training occurs in a repeated cycle called an **epoch**:

1. **Forward pass** — data flows through the network from input to output, producing a prediction $\hat{y}$
2. **Error calculation** — the **loss function** measures how wrong the prediction was: $\mathcal{L}(\hat{y}, y)$
3. **Backward pass** — the **backpropagation** algorithm computes the gradient of the loss with respect to each parameter using the chain rule from differential calculus
4. **Weight update** — the **optimizer** (e.g. Adam, SGD) adjusts the weights in the opposite direction of the gradient to reduce the error

$$w \leftarrow w - \eta \cdot \frac{\partial \mathcal{L}}{\partial w}$$

where $\eta$ is the **learning rate** — a hyperparameter that controls the size of the update step.

PyTorch automates steps 3 and 4 with `loss.backward()` and `optimizer.step()`.

### What does the partial derivative of the loss function mean?

**What is a partial derivative?**<br />
A derivative measures *"if I change this variable a little, how much does the function change?"*

When a function depends on **multiple variables**, the **partial** derivative with respect to one of them measures the effect of changing *only that one*, keeping all others fixed. The symbol $\partial$ (read "del") indicates this — in contrast to $d$ from the ordinary derivative.

**Partial with respect to what?**<br />
The loss function $\mathcal{L}$ depends on **all the network's weights** simultaneously — potentially thousands or millions of parameters: $w_1, w_2, \ldots, w_n$.

$$\frac{\partial \mathcal{L}}{\partial w}$$

means: *"keeping all other weights fixed, if I change this weight $w$ a little, how much does the loss increase or decrease?"*

The answer is a number — the **local gradient** of that weight. If positive, increasing $w$ increases the error; if negative, increasing $w$ decreases the error.

**Why subtract?**<br />
The **minus** sign is what does the trick: moving $w$ in the **opposite** direction of the gradient guarantees that the loss decreases. This is **gradient descent** — descending the "slope" of the error surface.

$\eta$ (learning rate) controls the step size — too large and the model "overshoots" the minimum; too small and convergence is slow.

### Further reading

For a visual and intuitive explanation of how each component works, the following is highly recommended:

- [But what is a neural network? — 3Blue1Brown](https://www.youtube.com/watch?v=aircAruvnKk)
- [Gradient descent, how neural networks learn — 3Blue1Brown](https://www.youtube.com/watch?v=IHZwWFHWa-w)
- [Backpropagation calculus — 3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8)

## Classification with MLP

### Dataset

The dataset used is the same as in the BERT fine-tuning notebook: the [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic).

Here, however, we will use the **numerical features** extracted from the images — radius, texture, perimeter, area, etc. — instead of the text reports. This makes the problem simpler and more direct, ideal for demonstrating the MLP without the complexity of natural language processing.

### Data preparation

Data preparation is covered in the [PrepareData](PrepareData.en.ipynb) notebook.

The data must be saved at `../data/breast_cancer.parquet` before running this notebook.

### Check available software and hardware

In [ ]:
import torch

print("PyTorch version: ", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
print("CUDA version: ", torch.version.cuda)

### Data acquisition

In [ ]:
import pandas as pd

# Numerical features used as network input
FEATURE_COLUMNS = [
    "radius", "texture", "perimeter", "area",
    "smoothness", "compactness", "concavity",
    "concave points", "symmetry", "fractal dimension"
]

def preprocess_dataset(path):
    df = pd.read_parquet(path)

    # Encode diagnosis: Malignant=1, Benign=0
    df['diagnosis_encoded'] = df['diagnosis'].apply(lambda x: 1 if x == 'M' else 0)

    return df

data = preprocess_dataset("../data/breast_cancer.parquet")

data[['id_number', 'diagnosis', 'diagnosis_encoded'] + FEATURE_COLUMNS].head()

### Diagnosis distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count of each class
data["diagnosis"].value_counts()

In [ ]:
sns.pairplot(data, hue='diagnosis', vars=['radius', 'texture', 'perimeter', 'area'], palette='Set2')
plt.suptitle("Attribute Distribution by Diagnosis", y=1.02)
plt.show()

### Data normalization

Features have very different scales (e.g. `area` can be 1000x larger than `smoothness`). Neural networks are sensitive to input scale — gradients from features with very large values would dominate learning.

**Normalization** solves this: `StandardScaler` subtracts the mean and divides by the standard deviation, making each feature have mean 0 and standard deviation 1.

> **Note:** the scaler must be fitted **only on the training data** and then applied to the validation data — to avoid *data leakage* (information from the test set leaking into the model).

### Train / validation split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data[FEATURE_COLUMNS].values
y = data['diagnosis_encoded'].values

# Split into train (80%) and validation (20%)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize: fit on train, apply to both train and validation
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)

print(f"Train:      {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Features:   {X_train.shape[1]}")

### Create Datasets and DataLoaders

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Convert NumPy arrays to PyTorch tensors
train_inputs  = torch.tensor(X_train, dtype=torch.float32)
train_labels  = torch.tensor(y_train, dtype=torch.long)
val_inputs    = torch.tensor(X_val,   dtype=torch.float32)
val_labels    = torch.tensor(y_val,   dtype=torch.long)

# Group features and labels into Datasets
train_dataset = TensorDataset(train_inputs, train_labels)
val_dataset   = TensorDataset(val_inputs,   val_labels)

batch_size = 16

# DataLoaders: iterate over the data in batches during training
train_dataloader = DataLoader(train_dataset, shuffle=True,  batch_size=batch_size)
val_dataloader   = DataLoader(val_dataset,   shuffle=False, batch_size=batch_size)

### Model definition

The architecture is intentionally small for didactic purposes:

```
Input (10 features)
    ↓
Linear layer:  10 → 16  +  ReLU
    ↓
Linear layer:  16 →  8  +  ReLU
    ↓
Linear layer:   8 →  2  (logits: Benign / Malignant)
```

**10 (input)**<br />
The 10 numerical features extracted from the images (radius, texture, perimeter, area, etc.).

**10 → 16 (expansion)**<br />
The network needs "room" to learn combinations of the 10 original features. With only 10 neurons in the hidden layer, each neuron would be roughly "responsible" for one feature — leaving no capacity to learn interactions between them (e.g. "large radius *and* irregular texture"). Going to 16 creates that extra space for richer intermediate representations.

**16 → 8 (compression)**<br />
After expanding, compressing forces the network to distill what it has learned. It must discard noise and retain only what truly discriminates between classes. This is an intentional bottleneck — similar to the encoder concept in autoencoders. Empirically, this "hourglass" pattern tends to generalize better than keeping a constant size.

**8 → 2 (output)**<br />
A raw score (logit) for each class.

**The general pattern is:**<br />
input → expansion → compression → output

**About the numbers**<br />
The specific numbers (16 and 8) are heuristics — powers of 2 are conventional (they align well with hardware) and a ~2x reduction factor per layer is a common rule of thumb. There is no exact formula; in production this would be a hyperparameter searched via cross-validation.

In PyTorch, models are defined as classes that inherit from `nn.Module`. The `forward` method describes how data flows through the network (the *forward pass*).

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 16),  # Layer 1: input_size → 16 neurons
            nn.ReLU(),                  # ReLU activation
            nn.Linear(16, 8),           # Layer 2: 16 → 8 neurons
            nn.ReLU(),                  # ReLU activation
            nn.Linear(8, 2)             # Output layer: 8 → 2 classes
        )

    def forward(self, x):
        return self.network(x)


input_size = len(FEATURE_COLUMNS)  # 10 features
model = MLP(input_size)

print(model)
print(f"\nTotal trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

### Optimizer and loss function

In [ ]:
# CrossEntropyLoss: combines Softmax + NLLLoss — ideal for multi-class classification
criterion = nn.CrossEntropyLoss()

# Adam: adaptive optimizer, generally converges faster than plain SGD
# lr (learning rate): size of the weight update step
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### Model training

In [ ]:
from tqdm import tqdm

# Select device: GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs = 10

# History for later visualization
history = {"loss": [], "val_accuracy": []}

for epoch in tqdm(range(epochs), desc="Training"):

    # ── Training ─────────────────────────────────────────────────────────────
    model.train()  # Enables training mode (activates dropout, batch norm, etc.)
    total_loss = 0

    for batch_inputs, batch_labels in train_dataloader:

        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)

        # 1. Zero the gradients accumulated from the previous step
        optimizer.zero_grad()

        # 2. Forward pass: compute predictions
        logits = model(batch_inputs)

        # 3. Compute the loss
        loss = criterion(logits, batch_labels)

        # 4. Backward pass: compute gradients via backpropagation
        loss.backward()

        # 5. Update weights with the optimizer
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataloader)

    # ── Validation ───────────────────────────────────────────────────────────
    model.eval()  # Disables training-only behaviours
    total_correct = 0
    total_samples = 0

    with torch.no_grad():  # Disables gradient computation (faster)
        for batch_inputs, batch_labels in val_dataloader:

            batch_inputs = batch_inputs.to(device)
            batch_labels = batch_labels.to(device)

            logits      = model(batch_inputs)
            predictions = torch.argmax(logits, dim=-1)

            total_correct += (predictions == batch_labels).sum().item()
            total_samples += batch_labels.size(0)

    val_accuracy = total_correct / total_samples

    history["loss"].append(avg_loss)
    history["val_accuracy"].append(val_accuracy)

    tqdm.write(f"Epoch {epoch + 1:02d}/{epochs}  |  Loss: {avg_loss:.4f}  |  Val Accuracy: {val_accuracy:.2%}")

### Learning curves

The learning curves show how the network evolved over epochs:

- **Loss (training)** — should decrease progressively. A smooth drop indicates the optimizer is finding a minimum in a stable way.
- **Accuracy (validation)** — may appear flat even as loss decreases, because accuracy is discrete (each mistake is worth ~0.88% on this dataset). What the loss is showing is that the model is becoming **more confident** in its correct predictions, not just getting more right.

In [ ]:
epoch_range = range(1, epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ── Loss ────────────────────────────────────────────────────────────────────
ax1.plot(epoch_range, history["loss"], color="steelblue", linewidth=2)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss per epoch (training)")
ax1.grid(True, alpha=0.3)

# ── Accuracy ─────────────────────────────────────────────────────────────────
ax2.plot(epoch_range, [v * 100 for v in history["val_accuracy"]], color="seagreen", linewidth=2)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy per epoch (validation)")
ax2.set_ylim(90, 101)
ax2.grid(True, alpha=0.3)

plt.suptitle("Learning Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Model evaluation

In [ ]:
y_test  = []
y_pred  = []
y_proba = []

model.eval()

with torch.no_grad():
    for batch_inputs, batch_labels in val_dataloader:

        batch_inputs = batch_inputs.to(device)
        batch_labels = batch_labels.to(device)

        logits = model(batch_inputs)

        y_test.extend(batch_labels.cpu().numpy())

        preds = torch.argmax(logits, dim=-1)
        y_pred.extend(preds.cpu().numpy())

        # Probability of the positive class (Malignant = 1)
        y_proba.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())

#### Accuracy, Precision, Recall and F1

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_proba)

print(f"Accuracy:  {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")
print(f"ROC AUC:   {roc_auc:.2f}")

#### Classification report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))

#### Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix

sns.heatmap(
    confusion_matrix(y_test, y_pred),
    annot=True, fmt='d', cmap='Blues',
    xticklabels=["Benign", "Malignant"],
    yticklabels=["Benign", "Malignant"]
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

#### Precision-Recall Curve

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_proba)

plt.plot(recall_curve, precision_curve)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()

#### ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], 'k--', label="Random")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

### Usage example

In [ ]:
import numpy as np

# New samples: [radius, texture, perimeter, area, smoothness,
#               compactness, concavity, concave_points, symmetry, fractal_dimension]
new_samples = np.array([
    # Typically benign profile: small radius, smooth texture
    [12.0, 15.0, 78.0, 450.0, 0.09, 0.07, 0.03, 0.02, 0.18, 0.06],
    # Typically malignant profile: large radius, irregular texture
    [22.0, 28.0, 145.0, 1500.0, 0.15, 0.25, 0.30, 0.15, 0.28, 0.09],
])

# Apply the same normalization used during training
new_samples_scaled = scaler.transform(new_samples)
new_tensor = torch.tensor(new_samples_scaled, dtype=torch.float32).to(device)

# Prediction
model.eval()
with torch.no_grad():
    logits      = model(new_tensor)
    probs       = torch.softmax(logits, dim=1)
    predictions = torch.argmax(logits, dim=-1)

label_map = {0: "Benign", 1: "Malignant"}

for i, (pred, prob) in enumerate(zip(predictions, probs)):
    print(f"Sample {i + 1}: {label_map[pred.item()]}  "
          f"(Benign: {prob[0]:.1%}  |  Malignant: {prob[1]:.1%})")